# 11.10 Prophet

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.10_prophet.ipynb)

Companion code for Section 11.10 of the book. Run the cells in order from top to bottom.

**Example 11.10.1**

In [ ]:
# pip install prophet
try:
    from prophet import Prophet
    from prophet.diagnostics import cross_validation, performance_metrics
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print('Prophet not installed. Install with: pip install prophet')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

# ── Prepare data in Prophet format ───────────────────────────────
# Prophet requires columns 'ds' (datetime) and 'y' (target)
np.random.seed(42)
n = 730
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

y = (1000 + 0.8*t + 300*np.sin(2*np.pi*t/365-np.pi/2)
     + 150*np.sin(2*np.pi*t/7) + np.random.normal(0,60,n))

df_prophet = pd.DataFrame({'ds': dates, 'y': y})

# Add external regressor
df_prophet['temperature'] = 20 + 15*np.sin(2*np.pi*t/365) + np.random.normal(0,3,n)

# Train-test split
train_p = df_prophet.iloc[:-90]
test_p  = df_prophet.iloc[-90:]

if PROPHET_AVAILABLE:
    # ── Fit Prophet model ─────────────────────────────────────────
    m = Prophet(
        changepoint_prior_scale=0.05,    # Flexibility of trend changes
        seasonality_prior_scale=10,       # Flexibility of seasonality
        holidays_prior_scale=10,
        seasonality_mode='additive',
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        uncertainty_samples=1000          # For credible intervals
    )

    # Add external regressor
    m.add_regressor('temperature')

    # Add custom seasonality (monthly)
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)

    m.fit(train_p)

    # Forecast
    future = m.make_future_dataframe(periods=90, freq='D')
    future['temperature'] = df_prophet['temperature'].values  # Must provide regressor
    forecast = m.predict(future)

    # Evaluate
    y_pred_p = forecast.set_index('ds').loc[test_p['ds'], 'yhat'].values
    mae_p    = mean_absolute_error(test_p['y'], y_pred_p)
    mape_p   = np.mean(np.abs((test_p['y'].values - y_pred_p) / test_p['y'].values))*100
    print(f'Prophet: MAE={mae_p:.2f}  MAPE={mape_p:.2f}%')

    # Component visualisation
    fig = m.plot_components(forecast)
    plt.suptitle('Prophet Components: Trend + Seasonalities', fontsize=13, fontweight='bold')
    plt.savefig('prophet_components.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Main forecast plot
    fig = m.plot(forecast)
    plt.suptitle('Prophet Forecast with Uncertainty Intervals',
                fontsize=13, fontweight='bold')
    plt.savefig('prophet_forecast.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Prophet cross-validation ──────────────────────────────────
    df_cv = cross_validation(
        m,
        initial='365 days',   # Minimum training period
        period='30 days',     # Spacing between cutoff dates
        horizon='90 days',    # Forecast horizon
        parallel='processes'
    )

    df_perf = performance_metrics(df_cv)
    print(f'\nProphet Cross-Validation Metrics (90-day horizon):')
    print(df_perf[['horizon','mae','mape','rmse','coverage']].tail(10).round(4))

    # ── Prophet hyperparameters ───────────────────────────────────
    print('\n=== Prophet Key Hyperparameters ===')
    params_guide = {
        'changepoint_prior_scale': '0.05 (default); higher=more flexible trend; range 0.001-0.5',
        'seasonality_prior_scale': '10 (default); higher=stronger seasonality; range 0.01-10',
        'holidays_prior_scale':    '10 (default); scales holiday effect size',
        'seasonality_mode':        'additive or multiplicative (use mult when seasonal ∝ trend)',
        'changepoint_range':       '0.8 (default); fraction of history where changepoints may occur',
        'n_changepoints':          '25 (default); max number of trend changepoints to detect',
        'interval_width':          '0.80 (default); width of uncertainty interval (80%)',
    }
    for param, desc in params_guide.items():
        print(f'  {param:<28}: {desc}')

else:
    print('Prophet not available - showing parameter guide only')
    print('Install: pip install prophet (requires pystan and cmdstanpy)')